# PhoWhisper LoRA fine-tune — Kaggle runner

Config-driven pipeline. `src/` never hardcodes a Kaggle path — this notebook is the
only place `/kaggle/input/...` appears, passed in via `--override`.

**Cell 1 clones the repo from GitHub `main`** — it does NOT use your local working
tree. Anything you changed locally (`configs/experiment.yaml`, `src/`, `scripts/`)
must be committed and pushed to `main` first, or this session runs the old code and
the run's frozen `config.json` records values you never intended.

**Before running**: attach as Kaggle Dataset inputs (Add Data):
- `paid-dataset-v2` (from `dataset/paid-dataset-v2/` in this repo, ~985 MB — consolidated
  2026-08-02 from the dot2 data drop + legacy test meetings repurposed as train, see
  `PROJECT_CORE.md` §4 and `scripts/ingest_paid_dataset_v2.py`. Zip and upload as a new
  Kaggle Dataset — this supersedes the old `paid-dataset` attachment.)
- `youtube-meetings` (from `dataset/youtube-meetings/` in this repo -- manifest + `audio/`
  only, **not** `raw/`; `audio/` alone is ~382 MB, the whole directory with `raw/` is
  1.35 GB, which is why `raw/` must be left out. Must be fully reviewed first:
  `scripts/review_youtube.py --check` must pass with no `verified: false` records.
  Merged with `paid-dataset-v2` by the cell in step 2 below into
  `/kaggle/working/dataset/mixed-noisy-v1`, see `youtube-data-pilot/README.md` step 6.)
- `real-meetings-bench` (from `dataset/real-meetings-bench/` in this repo, ~80 MB,
  produced by `scripts/ingest_real_bench.py` — zip and upload as a Kaggle Dataset)
- (optional) a GPU accelerator (T4 x1 is enough — v3-r16 ran `-large` at `lora.rank=16`
  with train batch 2 × grad_accum 8 for 801 steps in 6 h 31 m. `mixed-noisy-v1` has
  4717 train segments instead of 4270, i.e. 885 steps at the same effective batch, and
  every eval split is larger too — budget roughly 8 h and do not queue anything else
  into the same 12 h session.)

VIVOS (OOD) does not need a Kaggle Dataset attachment — `scripts/fetch_vivos.py`
downloads it directly from HF Hub in step 5. It is not optional: `src/config.py`
`validate()` now refuses a null `data.ood_eval_path`, because tier 2 is the only
measurement of forgetting in the gate and `select_lambda` has nothing to select
without an `ood_cer` column.

Run cells **in order**, stopping to read output at each stage before continuing —
this pipeline has never run end-to-end on `mixed-noisy-v1`; do not queue all cells blind.

## 1. Clone / update the repo

In [ ]:
import os

# Force single-GPU: on a T4 x2 session, Trainer/accelerate auto-wraps the model in
# legacy torch.nn.DataParallel when it sees >1 visible GPU without a distributed
# launch (accelerate launch / torchrun) -- that replicates the model and concentrates
# gradient reduction on one GPU, wasting memory for no speed benefit here. This
# pipeline is designed single-GPU only; set before any stage touches CUDA.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

REPO_URL = "https://github.com/egoist-minh/Reworkwhisper-finetune.git"
REPO_DIR = "/kaggle/working/Reworkwhisper-finetune"

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull origin main
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
os.environ["TRANSFORMERS_AUTO_CONVERSION"] = "0"

In [ ]:
!pip install -q -r requirements.txt

## 2. Locate attached datasets

Confirm the exact mount paths before setting the overrides below — Kaggle slugs the
dataset name, so this can differ from what you expect.

In [ ]:
!ls -la /kaggle/input

In [ ]:
# Merge paid-dataset-v2 (synthetic) with youtube-meetings (real, reviewed YouTube
# audio -- scripts/review_youtube.py --check must already pass) into one directory
# so configs/experiment.yaml:data.dataset_path can stay a single path. Kaggle mount
# paths nest one level deeper than the attached dataset name (Cell above exists to
# confirm this) -- edit both --sources paths to match what it printed. Run as a
# module (-m scripts.build_mixed_dataset), not a bare file path -- it imports
# scripts.review_youtube, same convention as
# youtube-data-pilot/style-guide.md's `python -m scripts.review_youtube`.
#
# --out is absolute (outside the cloned repo) and holds a real copy of both corpora,
# ~1.4 GB -- /kaggle/working is capped at 20 GB and also holds outputs/ and the
# checkpoints, so do not merge a second corpus into the same session.
# If this cell fails partway, `rm -rf /kaggle/working/dataset/mixed-noisy-v1` before
# re-running: _check_dest_empty refuses to merge on top of a non-empty destination.
# Expected split_stats on mixed-noisy-v1: train 4717 / val 365 / test 654.
!python -m scripts.build_mixed_dataset \
    --sources /kaggle/input/datasets/winhkento/paid-dataset-v2/paid-dataset-v2 \
              /kaggle/input/datasets/winhkento/youtube-meetings/youtube-meetings \
    --out /kaggle/working/dataset/mixed-noisy-v1

## 3. Set platform-specific paths

Edit these three to match what Cell above printed. This is the *only* place a
`/kaggle/input/...` path is written — everything downstream goes through
`--override`, never a hardcoded path inside `src/`.

In [ ]:
DATASET_PATH = "/kaggle/working/dataset/mixed-noisy-v1"       # written by the build_mixed_dataset cell in step 2
REAL_BENCH_PATH = "/kaggle/input/datasets/winhkento/real-meetings-bench/real-meetings-bench"  # edit to match step 2's listing
OOD_EVAL_PATH = "/kaggle/working/Reworkwhisper-finetune/dataset/vivos"  # written by the fetch_vivos cells in step 5
                                                                        # (relative --out dataset/vivos, cwd is the repo)

OVERRIDES = (
    f"--override data.dataset_path={DATASET_PATH} "
    f"--override data.real_bench_path={REAL_BENCH_PATH} "
    f"--override data.ood_eval_path={OOD_EVAL_PATH}"
)
print(OVERRIDES)

## 4. Model & hyperparameters

Every value below is a real field in `configs/experiment.yaml`, applied via
`--override` — this isn't a new config surface, just a convenient place to see and
change what a run actually uses instead of hand-editing YAML or writing override
strings from scratch.

**`EVAL_LIMIT` — read this before trusting any number from a prior run.**
`configs/experiment.yaml` ships `eval.limit: null` (fixed 2026-08-02 — used to be
`20`, an early smoke-testing leftover). `src/gate.py`'s `_eval_split` applies this to
*every* eval — baseline **and** every sweep-gate tier. On `mixed-noisy-v1` the full
splits are test **654** (426 synthetic + 228 YouTube) / val **365** (250 + 115) /
VIVOS 760 / real-bench 264. `EVAL_LIMIT` below should stay `null` for a trustworthy
run.

**Decode cost, since every one of those splits got bigger.** Per epoch, training
evaluates val (365) + VIVOS (760). The sweep decodes val + VIVOS once per λ, i.e.
5 × 1125. The gate then decodes test (654) twice, VIVOS once, real-bench twice —
the doubled ones are `normalization_check`, which runs whenever
`normalization.audit_conversions` is true. Beam is 1 (`eval.num_beams`); leave it
there.

In [ ]:
BASE_MODEL = "vinai/PhoWhisper-large"    # 1.64B params, 28.8M trainable at rank 16 (1.76%),
                                          # measured on the v3-r16 run. There is no
                                          # PhoWhisper "large-v3" on the Hub -- the v3 in this
                                          # project's names is the model line (reworkwhisper-large-v3),
                                          # not a base-model variant.

LORA_RANK = 16
LORA_ALPHA = 32

TRAIN_EPOCHS = 3
TRAIN_BATCH_SIZE = 2              # MEASURED on -large: v3-r16 ran 2 x 8 accum for 801 steps in
                                   # 6 h 31 m (29.34 s/step) on one T4 without OOM. The 9.75 GiB
                                   # @ batch 8 figure in CLAUDE.md is -small, not this model --
                                   # raising this is an untested change, and mixed-noisy-v1's
                                   # YouTube labels are ~3x longer than synthetic ones (mean 220
                                   # vs 80 chars), so decoder activations grow too. Watch
                                   # nvidia-smi if you do raise it.
GRAD_ACCUM_STEPS = 8              # effective batch 16 -> 4717/16 = 295 steps/epoch, 885 total
LEARNING_RATE = 2.0e-4
TRAIN_LIMIT = "null"
EVAL_LIMIT ="null"# null = full split. See markdown above -- was left at 20 (smoke-test value)
EVAL_BATCH_SIZE = 8

SWEEP_LAMBDAS = "[0.0,0.25,0.5,0.75,1.0]"
OOD_CER_BUDGET = 0.02
REAL_CER_REGRESSION_PP = 0.0     # tier 4a zero-tolerance -- loosen only with a deliberate decision, see SESSIONS.md

PARAM_OVERRIDES = (
    f"--override base_model={BASE_MODEL} "
    f"--override lora.rank={LORA_RANK} "
    f"--override lora.alpha={LORA_ALPHA} "
    f"--override training.epochs={TRAIN_EPOCHS} "
    f"--override training.batch_size={TRAIN_BATCH_SIZE} "
    f"--override training.grad_accum_steps={GRAD_ACCUM_STEPS} "
    f"--override training.limit={TRAIN_LIMIT} "
    f"--override training.learning_rate={LEARNING_RATE} "
    f"--override eval.limit={EVAL_LIMIT} "
    f"--override eval.batch_size={EVAL_BATCH_SIZE} "
    f"--override 'sweep.lambdas={SWEEP_LAMBDAS}' "
    f"--override sweep.ood_cer_budget={OOD_CER_BUDGET} "
    f"--override gates.real_cer_regression_pp={REAL_CER_REGRESSION_PP}"
)

# Re-running this cell appends PARAM_OVERRIDES a second time -- harmless (last value
# wins) but confusing to read. Re-run the path cell in step 3 first if you edit here.
OVERRIDES = OVERRIDES + " " + PARAM_OVERRIDES
print(OVERRIDES)

## 5. Fetch VIVOS (OOD benchmark)

**Untested end-to-end before this run** — parquet route primary, tarball fallback.
Read the printed schema before trusting the manifest it writes.

In [ ]:
!python scripts/fetch_vivos.py --out dataset/vivos --smoke

In [ ]:
# If the smoke run above looks right, fetch the full test split (no --smoke / --limit):
!python scripts/fetch_vivos.py --out dataset/vivos

## 6. Stage: smoke

CPU-only, no model download. Proves config load, manifest merge, split resolution,
normalization, and the peft compat patch all work on this exact Kaggle image before
any GPU time is spent. **This has never run on `mixed-noisy-v1` before — read the
output carefully, do not assume it just works.**

Expected `split_stats`: `{'train': 4717, 'val': 365, 'test': 654}`. Anything else
means the merge or `data.val_meetings` is not what you think — `val_meetings` must
list `rCd8DSMk3-c` alongside the three synthetic meetings, or val holds no YouTube
audio at all and λ\* gets picked blind to the slice this run exists to learn.

In [ ]:
!python -m src.pipeline --stage smoke {OVERRIDES}

### 6b. Pre-flight: label length vs the decoder's limit

New for `mixed-noisy-v1`. `WhisperCollator` (`src/train.py`) tokenizes labels with
`padding=True` and **no truncation**, so a label longer than the decoder's
`max_target_positions` would only surface as a CUDA-side error partway into
training. YouTube labels are much longer than synthetic ones (up to 426 characters
vs 275), so check it here — tokenizer only, no model weights, a few seconds on CPU.

In [ ]:
import glob
import json

from transformers import WhisperConfig, WhisperTokenizer

tokenizer = WhisperTokenizer.from_pretrained(BASE_MODEL)
max_target_positions = WhisperConfig.from_pretrained(BASE_MODEL).max_target_positions

records = [json.loads(line)
           for path in sorted(glob.glob(f"{DATASET_PATH}/manifest.*.jsonl"))
           for line in open(path, encoding="utf-8") if line.strip()]
lengths = sorted(((len(tokenizer(r["text"]).input_ids), r["meeting_id"], r["segment_id"])
                  for r in records), reverse=True)

print(f"max_target_positions = {max_target_positions}")
for n_tokens, meeting_id, segment_id in lengths[:5]:
    print(f"  {n_tokens:4d} tokens  {meeting_id}/{segment_id}")

over_limit = [x for x in lengths if x[0] > max_target_positions]
if over_limit:
    raise SystemExit(
        f"{len(over_limit)} label(s) exceed {max_target_positions} tokens and "
        "WhisperCollator does not truncate -- re-segment those before training"
    )
print(f"OK: {len(records)} labels, longest {lengths[0][0]} tokens")

## 7. Stage: baseline

Base model over test + OOD + real bench. Writes `metrics/baseline.json` and
`audit/predictions_baseline_*.csv`. **Only run this after Cell 6 (smoke) is clean.**

In [ ]:
RUN_ID = "v4-mixed-r16"  # matches configs/experiment.yaml:run_id unless overridden here
!TRANSFORMERS_AUTO_CONVERSION=0 python -m src.pipeline --stage baseline --override run_id={RUN_ID} {OVERRIDES}

In [ ]:
import json
print(json.dumps(json.load(open(f"outputs/{RUN_ID}/metrics/baseline.json")), indent=2))

## 7b. LR probe — optional, and run it in its own session

There is no LR finder in HuggingFace Trainer (no equivalent of Lightning's
`Tuner.lr_find()`), so `scripts/lr_probe.py` is the manual substitute: it calls
`--stage train` once per candidate learning rate with `training.limit` capping the
split, then overlays the resulting loss curves. Read the three shapes off the plot —
too low is a straight slow decline, right is a fast drop that flattens, too high
bounces or rises.

It reuses this run's `validated_manifest.jsonl`, so **Cell 7 (baseline) must have
finished first**. Each probe writes a full `outputs/{RUN_ID}-lr{lr}/` tree, checkpoint
included; a probe whose `training.csv` already exists is skipped rather than repeated,
so an interrupted session resumes where it stopped.

**Budget it as a separate session.** At `TRAIN_BATCH_SIZE=2` × `GRAD_ACCUM_STEPS=8`
the effective batch is 16, so `LR_PROBE_LIMIT=1600` is ~100 optimizer steps ≈ 49 min
per LR at the 29.34 s/step measured on v3-r16 — before the eval below. Three probes
plus the ~7 h real run does not fit in one Kaggle session.

**The eval tax does not shrink with `LR_PROBE_LIMIT`.** `training.limit` caps train,
val and OOD together (`src/pipeline.py:192`), and val (365) + VIVOS (760) are both
already under 1600, so every probe still decodes 1125 segments at the end of its
epoch. That eval cannot be skipped — `src/train.py` raises if `eval_val_cer` is never
observed, which is deliberate.

Once a winner is clear, set `LEARNING_RATE` in Cell 4 above, **re-run that cell**, then
go on to Cell 8.

In [ ]:
LR_PROBE_LRS = "5e-5,2e-4,6e-4"   # bracketing LEARNING_RATE above. use_rslora=true keeps
                                  # scaling at alpha/sqrt(rank) = 8.0, the same as v3-r16 --
                                  # if you turn rsLoRA off, scaling drops to alpha/rank = 2.0
                                  # and this grid should shift up ~4x.
LR_PROBE_LIMIT = 1600             # segments; 1600 / 16 effective batch = ~100 optimizer steps
LR_PROBE_EPOCHS = 1

# Built as an argv list and run through subprocess rather than `!python ... {VAR}`:
# the `{}` substitution in a `!` line is an IPython feature, and it passed the
# braces through literally here (`--limit: invalid int value: '{LR_PROBE_LIMIT}'`).
# An argv list has no substitution step to fail, and no quoting to get wrong --
# OVERRIDES reaches --extra-args as one argument even though it contains spaces.
#
# OVERRIDES is forwarded so the probe trains on the same paths, model, batch size
# and rank as the real run. The script's own run_id / learning_rate / limit /
# epochs are applied last and win over the copies inside OVERRIDES.
import shlex
import subprocess

cmd = ["python", "scripts/lr_probe.py",
       "--source-run", f"outputs/{RUN_ID}",
       "--lrs", LR_PROBE_LRS,
       "--limit", str(LR_PROBE_LIMIT),
       "--epochs", str(LR_PROBE_EPOCHS),
       "--extra-args", OVERRIDES]
print(shlex.join(cmd), "\n")
subprocess.run(cmd, check=True)

In [ ]:
from IPython.display import Image, display

print(open("outputs/lr-probe/summary.csv").read())
display(Image("outputs/lr-probe/loss_curves.png"))

## 8. Stage: train

LoRA SFT, rank from `configs/experiment.yaml` (16, or `LORA_RANK` above if you
changed it). Both of the things that were unverified before v3-r16 have since run on
a real T4 and worked — `Trainer(eval_dataset=dict)` with `{"val":..,"ood":..}`, and
the custom `RobustEvalTrackingCallback` that replaced the built-in
`EarlyStoppingCallback` / `load_best_model_at_end` (`src/train.py`). It saves
`checkpoints/best/` itself the instant `eval_val_cer` improves, and raises loudly if
`eval_val_cer` is never observed rather than silently shipping a wrong checkpoint.
Still confirm `checkpoints/best/` appears partway through, not only at the end.

**What to watch that is new on `mixed-noisy-v1`:** the `ValCER` column is now a pooled
number over 250 synthetic + 115 YouTube segments, of which YouTube is 63% of the
characters — best-checkpoint selection therefore tracks the YouTube slice more than
the synthetic one. `OOD_CER` (VIVOS, every eval round) is the forgetting signal: on
v3-r16 it climbed 0.0559 → 0.0494 → 0.0420 across epochs against a 0.0228 baseline,
so a rising column is expected; the λ sweep is what brings it back inside budget.

In [ ]:
!python -m src.pipeline --stage train --override run_id={RUN_ID} {OVERRIDES}

## 9. HF token (only needed if you intend to push in Cell 10)

Add `HF_TOKEN` under this notebook's Add-ons → Secrets first. Never hardcode the
token here — it must not end up in any committed artifact.

Move/run this cell earlier (right after Cell 2) if you want it set for every stage
— that also silences the "unauthenticated requests to the HF Hub" warning that
otherwise appears on every stage's model load.

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN set")
except Exception as e:
    print(f"No HF_TOKEN secret configured ({e}) -- fine if hub.push is false")

## 10. Stage: sweep-gate

λ sweep (hard-fails if no λ fits `sweep.ood_cer_budget` — no fallback) → gate tiers
1/2/4a → HF push iff `overall_pass` and `hub.push: true`. Set `hub.push`/`hub.repo_id`
below only once you've decided to actually publish — this is an outward-facing action.

**Tier 1 is now judged per source as well as pooled (2026-08-16).** The test split is
59.6% YouTube characters from 34.9% of the segments, so a single pooled bound lets a
large YouTube gain hold the tier open while the synthetic slice regresses. Each slice
in `tier1_in_domain.by_source` now carries its own `bound`/`pass` (tier 1's own rule
against that slice's baseline), and the tier passes only if the pooled bound and every
slice pass. A slice with no baseline available carries no `pass` and cannot fail the
tier — read the `verdict` string in that case.

**Tier 4a reports more than pass/fail (added 2026-08-02)** — `gate_results.json`'s
`tier4a_real` carries `by_meeting` (CER per real recording, don't just read the pooled
number), `delta_ci`/`verdict` (paired comparison vs baseline on the same segments —
read `verdict`: `INCONCLUSIVE` means the ~264-segment sample can't resolve the
difference, treat that as "no evidence," not as a pass), and `normalization_check`
(if the two number-convention CERs differ a lot, the result is normalization-driven,
not model-driven). Of these only the per-source `pass` above feeds `overall_pass` —
read the rest alongside the gate verdict, not instead of it.

**λ\* deserves a look, not trust.** `sweep.elbow_ratio_threshold: 10.0` was calibrated
on v3-r16's own sweep (8.9× step ratio at λ=0.5 accepted, 12.6× at λ=0.75 rejected).
Val has a different composition now, so read `metrics/lambda_sweep.csv` — including its
`val_cer_synthetic` / `val_cer_youtube` columns — before accepting the selection.

In [ ]:
HUB_PUSH = False        # flip to True only when ready to publish
HUB_REPO_ID = None       # e.g. "your-username/phowhisper-lora-v0-r16"

hub_overrides = f"--override hub.push={HUB_PUSH} " + (f"--override hub.repo_id={HUB_REPO_ID} " if HUB_REPO_ID else "")
!python -m src.pipeline --stage sweep-gate --override run_id={RUN_ID} {OVERRIDES} {hub_overrides}

In [ ]:
import json
print(json.dumps(json.load(open(f"outputs/{RUN_ID}/metrics/gate_results.json")), indent=2))

In [ ]:
# On v3-r16, tqdm's redraw overwrote the last printed line, so the sweep table and
# the selected lambda* never showed up anywhere in that notebook -- only in the CSV.
# Print it here so this run's record isn't the same blind spot.
import csv
with open(f"outputs/{RUN_ID}/metrics/lambda_sweep.csv") as f:
    for row in csv.DictReader(f):
        print(row)

## 11. Evidence — CER + predictions

Everything under `outputs/{run_id}/` is the run's evidence: `metrics/baseline.json`,
`metrics/lambda_sweep.csv`, `metrics/gate_results.json`, and every
`audit/predictions_*.csv` (segment-level ref/hyp for baseline and gate, per tier).
Download this whole folder before the Kaggle session ends — it is not saved anywhere
else.

In [ ]:
!find outputs/{RUN_ID} -type f | sort

In [ ]:
!zip -r -q outputs_{RUN_ID}.zip outputs/{RUN_ID}

from IPython.display import FileLink
FileLink(f"outputs_{RUN_ID}.zip")